In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
!pip install plyfile

In [18]:
%run /content/drive/MyDrive/Pi3/example.py

Loading frames from video: /content/drive/MyDrive/Pi3/examples/muffinO.mp4
Found 14 images/frames. Processing...
All images will be resized to a uniform size: (672, 378)
Detecting walls via Distance Clustering...
Walls removed. Removed 133310 points.
Done


In [ ]:
pip install open3d

  Using cached open3d-0.19.0-cp312-cp312-manylinux_2_31_x86_64.whl.metadata (4.3 kB)
  Using cached dash-3.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached configargparse-1.7.1-py3-none-any.whl.metadata (24 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached addict-2.4.0-py3-none-any.whl.metadata (1.0 kB)
  Using cached pyquaternion-0.9.9-py3-none-any.whl.metadata (1.4 kB)
  Using cached retrying-1.4.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached comm-0.2.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jedi-0.19.2-py2.py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 145.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 103.0 MB/s eta 0:00:00
  

load the PLY file, remove floaters using `remove_statistical_outliers`, and save the result. Might need to adjust `nb_neighbors` and `std_ratio` parameters based on specific point cloud data for optimal floater removal.

In [ ]:
import open3d as o3d
import os

def remove_floaters_from_ply(input_ply_path, output_ply_path, nb_neighbors=20, std_ratio=.2):
    """
    Reads a PLY point cloud, removes statistical outliers (floaters), and saves the cleaned point cloud.

    Args:
        input_ply_path (str): Path to the input PLY file.
        output_ply_path (str): Path to save the cleaned PLY file.
        nb_neighbors (int): Number of neighbors to consider for statistical outlier removal.
        std_ratio (float): Standard deviation ratio threshold. Points with a distance
                           mean further than this are removed.
    """
    print(f"Loading point cloud from: {input_ply_path}")
    if not os.path.exists(input_ply_path):
        print(f"Error: Input file not found at {input_ply_path}")
        return

    pcd = o3d.io.read_point_cloud(input_ply_path)

    if not pcd.has_points():
        print(f"Error: Loaded point cloud from {input_ply_path} contains no points.")
        return

    print(f"Original point cloud has {len(pcd.points)} points.")

    # Apply statistical outlier removal
    print(f"Removing floaters using statistical outlier removal (nb_neighbors={nb_neighbors}, std_ratio={std_ratio})...")
    cl, ind = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)
    cleaned_pcd = pcd.select_by_index(ind)

    print(f"Cleaned point cloud has {len(cleaned_pcd.points)} points (removed {len(pcd.points) - len(cleaned_pcd.points)} floaters).")

    print(f"Saving cleaned point cloud to: {output_ply_path}")
    o3d.io.write_point_cloud(output_ply_path, cleaned_pcd)
    print("Done.")

# --- Example Usage ---
# Define your input and output paths here
input_file = "/content/drive/MyDrive/PointClouds3D/shoeTest2Conf.ply" # Replace with your input .ply file
output_file = "/content/drive/MyDrive/PointClouds3D/shoeTest2Conf_cleaned2.ply" # Replace with your desired output .ply file

remove_floaters_from_ply(input_file, output_file)


In [ ]:
# google_drive_storage.py
import os
import json
import pickle
import numpy as np
from io import BytesIO
from datetime import datetime
from typing import Dict, Any, Optional, List

# Google API Libraries
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build, Resource
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload
from googleapiclient.errors import HttpError

# If modifying these scopes, delete the token.pickle file.
SCOPES = ['https://www.googleapis.com/auth/drive.file']
DEFAULT_FOLDER = 'PointClouds3D'

class GoogleDriveStorage:
    """
    A class to store and manage 3D data (point clouds, meshes) and other files
    in a dedicated folder on Google Drive.
    """
    def __init__(self, credentials_file: str = '/content/drive/MyDrive/Token/credentials.json', token_file: str = '/content/drive/MyDrive/Token/token.pickle', verbose: bool = False):
        """
        Initializes and authenticates with the Google Drive API.

        Args:
            credentials_file: Path to the OAuth credentials JSON file.
            token_file: Path to store/load the authentication token.
            verbose: If True, prints detailed status messages.
        """
        self.credentials_file = credentials_file
        self.token_file = token_file
        self.verbose = verbose
        self.service = self._authenticate()

        self.folder_id = self._find_or_create('folder', DEFAULT_FOLDER)
        self.metadata_file_id = self._find_or_create(
            'file',
            'metadata_index.json',
            parents=[self.folder_id],
            initial_content=b'{}'
        )
        self.metadata_cache = self._load_metadata()

    def _log(self, message: str):
        """Prints a message if verbosity is enabled."""
        if self.verbose:
            print(message)

    def _authenticate(self) -> Resource:
        """Handles Google Drive authentication flow."""
        creds = None
        if os.path.exists(self.token_file):
            with open(self.token_file, 'rb') as token:
                creds = pickle.load(token)

        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                self._log("Refreshing access token...")
                creds.refresh(Request())
            else:
                self._log("Requesting new access token...")
                flow = InstalledFlow.from_client_secrets_file(self.credentials_file, SCOPES)
                # Use run_console() for headless environments like Colab
                creds = flow.run_console()

            with open(self.token_file, 'wb') as token:
                pickle.dump(creds, token)
                self._log(f"Credentials saved to {self.token_file}")

        return build('drive', 'v3', credentials=creds)

    def _find_or_create(self, item_type: str, name: str, parents: Optional[list] = None, initial_content: Optional[bytes] = None) -> str:
        """
        Finds a file or folder by name, creating it if it doesn't exist.

        Args:
            item_type: 'file' or 'folder'.
            name: The name of the item.
            parents: A list of parent folder IDs.
            initial_content: Initial content for a new file.

        Returns:
            The Google Drive ID of the item.
        """
        mime_types = {
            'folder': 'application/vnd.google-apps.folder',
            'file': 'application/json'
        }
        query = f"name='{name}' and mimeType='{mime_types[item_type]}' and trashed=false"
        if parents:
            query += f" and '{parents[0]}' in parents"

        results = self.service.files().list(q=query, spaces='drive', fields='files(id)').execute()
        items = results.get('files', [])

        if items:
            self._log(f"Found existing {item_type}: {name}")
            return items[0]['id']

        self._log(f"Creating new {item_type}: {name}...")
        file_metadata = {'name': name, 'mimeType': mime_types[item_type]}
        if parents:
            file_metadata['parents'] = parents

        media = None
        if item_type == 'file' and initial_content is not None:
             media = MediaIoBaseUpload(BytesIO(initial_content), mimetype=mime_types['file'])

        file = self.service.files().create(body=file_metadata, media_body=media, fields='id').execute()
        return file.get('id')

    def _load_metadata(self) -> Dict[str, Any]:
        """Loads the metadata index file from Google Drive."""
        try:
            buffer = self._download_media_to_buffer(self.metadata_file_id)
            return json.loads(buffer.read().decode('utf-8'))
        except (HttpError, json.JSONDecodeError) as e:
            self._log(f"Could not load metadata, starting fresh. Error: {e}")
            return {}

    def _save_metadata(self):
        """Saves the current metadata cache to Google Drive."""
        buffer = BytesIO(json.dumps(self.metadata_cache, indent=2).encode('utf-8'))
        media = MediaIoBaseUpload(buffer, mimetype='application/json', resumable=True)
        self.service.files().update(fileId=self.metadata_file_id, media_body=media).execute()

    def _download_media_to_buffer(self, file_id: str) -> BytesIO:
        """Downloads a file's content into a BytesIO buffer."""
        request = self.service.files().get_media(fileId=file_id)
        buffer = BytesIO()
        downloader = MediaIoBaseDownload(buffer, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                self._log(f"  Download progress: {int(status.progress() * 100)}%")
        buffer.seek(0)
        return buffer


    def delete(self, name: str):
        """Deletes an object and its metadata from Google Drive."""
        if name not in self.metadata_cache:
            raise ValueError(f"Object '{name}' not found.")

        file_id = self.metadata_cache[name]['file_id']
        try:
            self.service.files().delete(fileId=file_id).execute()
            del self.metadata_cache[name]
            self._save_metadata()
            self._log(f"Successfully deleted '{name}'.")
        except HttpError as error:
            print(f"An error occurred while deleting '{name}': {error}")
            if error.resp.status == 404:
                del self.metadata_cache[name]
                self._save_metadata()
            else:
                raise

    def list_from_cache(self) -> Dict[str, Any]:
        """Lists all items tracked in the local metadata cache."""
        return self.metadata_cache

    def list_drive_folder_contents(self) -> List[Dict[str, Any]]:
        """
        Lists all files/subfolders directly from the Google Drive folder via an API call.
        This provides a live view and can be used to find orphaned files.
        Handles pagination for folders with many files.

        Returns:
            A list of file resource dictionaries from the Google Drive API.
        """
        all_files = []
        page_token = None
        query = f"'{self.folder_id}' in parents and trashed=false"
        try:
            while True:
                response = self.service.files().list(
                    q=query,
                    spaces='drive',
                    fields='nextPageToken, files(id, name, size, mimeType, createdTime)',
                    pageToken=page_token
                ).execute()

                all_files.extend(response.get('files', []))
                page_token = response.get('nextPageToken', None)
                if page_token is None:
                    break
            self._log(f"Found {len(all_files)} items in the Drive folder '{DEFAULT_FOLDER}'.")
            return all_files
        except HttpError as error:
            print(f"An error occurred listing folder contents: {error}")
            return []

In [ ]:
import subprocess
import argparse
import shlex
import os
import time
import queue
import threading

# Define the path to the command file
COMMAND_FILE = '/content/drive/MyDrive/PointClouds3D/command_queue.txt'
# Assuming GoogleDriveStorage class is defined in a previous cell or imported
# from google_drive_storage import GoogleDriveStorage
storage_manager = GoogleDriveStorage(verbose=True) # Initialize if needed

def process_command(command_string):
    """
    Parses a command string and executes the corresponding action (run example, rename, delete).

    Args:
        command_string: The user-provided command string.
    """
    # Split the command string into command and arguments
    parts = shlex.split(command_string)
    if not parts:
        print("Warning: Received empty command.")
        return

    command = parts[0].lower()
    args = parts[1:]

    if command == 'run_example':
        # Pass the original arguments string directly to run_example_script
        run_example_script(' '.join(args)) # Pass the remaining parts as a single string
    elif command == 'rename':
        if len(args) == 2:
            old_name = args[0]
            new_name = args[1]
            rename_file(old_name, new_name)
        else:
            print("Error: Rename command requires two arguments: <old_name> <new_name>")
    elif command == 'delete':
        if len(args) == 1:
            file_identifier = args[0] # Changed from file_name to file_identifier
            delete_file(file_identifier) # Pass the identifier to delete_file
        else:
            print("Error: Delete command requires one argument: <file_identifier> (name or full path)") # Updated message
    else:
        print(f"Unknown command: {command}")


def run_example_script(command_string):
    """
    Parses arguments and runs example.py script.

    Args:
        command_string: The arguments string for example.py.
    """
    parser = argparse.ArgumentParser(description='Run the example.py script with specified arguments.')
    parser.add_argument('--data_path', help='Path to the input data.')
    parser.add_argument('--save_path', help='Path to save the output.')
    parser.add_argument('--interval', type=int, help='Sampling interval.')

    try:
        # Use shlex.split to handle quoted arguments correctly
        args = parser.parse_args(shlex.split(command_string))

        # Construct the base command
        cmd = ['python', '/content/drive/MyDrive/Pi3/example.py']

        # Add parsed arguments if they exist
        if args.data_path:
            cmd.extend(['--data_path', args.data_path])
        if args.save_path:
            cmd.extend(['--save_path', args.save_path])
        if args.interval is not None:
            cmd.extend(['--interval', str(args.interval)])

        print(f"Running example.py with command: {' '.join(cmd)}")

        # Execute the command using subprocess
        result = subprocess.run(cmd, capture_output=True, text=True, check=True, timeout=600)

        print("STDOUT:")
        print(result.stdout)
        print("STDERR:")
        print(result.stderr)

    except FileNotFoundError:
        print("Error: example.py script not found.")
    except subprocess.CalledProcessError as e:
        print(f"Error executing script: {e}")
        print("STDOUT:")
        print(e.stdout)
        print("STDERR:")
        print(e.stderr)
    except SystemExit:
        # argparse raises SystemExit on error or --help
        print("Error parsing arguments for example.py. Please check your command format.")
    except subprocess.TimeoutExpired:
        print(f"Error: Command timed out after 600 seconds.")
    except Exception as e:
        print(f"An unexpected error occurred while running example.py: {e}")


def rename_file(old_name, new_name):
    """Renames a file on Google Drive using GoogleDriveStorage."""
    try:
        print(f"Attempting to rename '{old_name}' to '{new_name}' using GoogleDriveStorage...")

        # Find the file ID for old_name in metadata or by listing drive contents
        file_id_to_rename = None
        # First, check the metadata cache
        for item_name, metadata in storage_manager.list_from_cache().items():
            # Check both the logical name used as key and the file_name property
            if item_name == old_name or metadata.get('file_name') == old_name:
                 file_id_to_rename = metadata.get('file_id')
                 # Optionally update the logical name in cache if it matched file_name
                 if item_name != old_name:
                     storage_manager.metadata_cache[new_name] = storage_manager.metadata_cache.pop(item_name)
                 break

        # If not found in cache, try listing directly from Drive (might be slower)
        if file_id_to_rename is None:
             drive_files = storage_manager.list_drive_folder_contents()
             for f in drive_files:
                 if f.get('name') == old_name:
                     file_id_to_rename = f.get('id')
                     # Add to cache if found
                     storage_manager.metadata_cache[new_name] = {
                         'file_id': file_id_to_rename,
                         'file_name': new_name,
                         'file_size': f.get('size'),
                         'created_at': f.get('createdTime'),
                         'custom_metadata': {} # Default empty
                     }
                     break


        if file_id_to_rename:
            storage_manager.service.files().update(fileId=file_id_to_rename, body={'name': new_name}).execute()
            print(f"Successfully renamed file ID {file_id_to_rename} ('{old_name}') to '{new_name}'.")

            # Update metadata cache - if the old_name was a key, rename it
            if old_name in storage_manager.metadata_cache:
                # Ensure the file_name property is also updated
                if 'file_name' in storage_manager.metadata_cache[old_name]:
                    storage_manager.metadata_cache[old_name]['file_name'] = new_name
                # If the key itself was the old_name, rename the key
                storage_manager.metadata_cache[new_name] = storage_manager.metadata_cache.pop(old_name)


            storage_manager._save_metadata() # Save the updated metadata
        else:
            print(f"Error: File '{old_name}' not found in tracked items or Drive folder.")

    except Exception as e:
        print(f"Error renaming file: {e}")


def delete_file(file_identifier):
    """Deletes a file on Google Drive using GoogleDriveStorage."""
    try:
        print(f"Attempting to delete '{file_identifier}' using GoogleDriveStorage...")

        file_id_to_delete = None
        metadata_key_to_delete = None

        # 1. Check metadata cache first, matching by key or 'file_name' value
        for item_name, metadata in storage_manager.list_from_cache().items():
            # Check if the identifier matches the logical name or the stored file_name
            if item_name == file_identifier or metadata.get('file_name') == file_identifier:
                file_id_to_delete = metadata.get('file_id')
                metadata_key_to_delete = item_name
                break

        # 2. If not found in cache, try listing directly from Drive, matching by name (handling full path)
        if file_id_to_delete is None:
            # Extract just the filename if a full path is provided
            file_name_only = os.path.basename(file_identifier)
            drive_files = storage_manager.list_drive_folder_contents()
            print(f"Files found when listing Drive folder contents:") # Added logging
            for f in drive_files:
                print(f"  - {f.get('name')}") # Added logging
                if f.get('name') == file_name_only: # Match using just the filename
                    file_id_to_delete = f.get('id')
                    metadata_key_to_delete = None # Not in cache, no key to delete
                    # Optional: Add to cache if found on Drive? Depends on desired behavior.
                    # For now, just find the ID for deletion.
                    break

        if file_id_to_delete:
             storage_manager.service.files().delete(fileId=file_id_to_delete).execute()
             print(f"Successfully deleted file ID {file_id_to_delete} ('{file_identifier}').")

             # Remove from metadata cache if it was found there
             if metadata_key_to_delete and metadata_key_to_delete in storage_manager.metadata_cache:
                 del storage_manager.metadata_cache[metadata_key_to_delete]
                 storage_manager._save_metadata() # Save the updated metadata
        else:
             print(f"Error: File '{file_identifier}' not found in tracked items or Drive folder.")

    except Exception as e:
        print(f"Error deleting file: {e}")


def get_command_from_file(file_path):
    """
    Reads the first command from a file, removes it, and returns it.
    Returns the command string or None if the file is empty or doesn't exist.
    """
    commands = []
    # Add a small delay to allow file writes to complete
    time.sleep(0.1)
    if os.path.exists(file_path):
        try:
            # Use a lock or file-level locking if multiple processes might access the file
            with open(file_path, 'r') as f:
                commands = f.readlines()

            if not commands:
                return None

            # Get the first command and remove leading/trailing whitespace
            command_to_process = commands[0].strip()

            # Remove the first command from the list
            remaining_commands = commands[1:]

            # Rewrite the file with the remaining commands
            with open(file_path, 'w') as f:
                f.writelines(remaining_commands)

            return command_to_process
        except Exception as e:
            print(f"Error reading/writing command file: {e}")
            return None
    else:
         return None


# Main loop to continuously check the command file
if __name__ == "__main__":
    print(f"Starting to monitor command file: {COMMAND_FILE}")
    print("Ensure your web app writes commands to this file, one per line.")
    print("Supported commands: run_example <args for example.py>, rename <old_name> <new_name>, delete <file_identifier>")


    # Thread for processing commands from the file
    def process_file_commands():
        while True:
            command_string = get_command_from_file(COMMAND_FILE)
            if command_string:
                print(f"Received command from file: {command_string}")
                process_command(command_string) # Use the new process_command function
            else:
                # print("No command in file. Waiting...") # Avoid excessive printing
                pass # Keep the loop running

            time.sleep(5) # Poll the file every 5 seconds

    # Start the file processing thread
    file_processing_thread = threading.Thread(target=process_file_commands, daemon=True)
    file_processing_thread.start()

    # Keep the main thread alive (optional, depending on how you want to manage the script's lifecycle)
    try:
        while True:
            time.sleep(1) # Small sleep to prevent high CPU usage in the main thread
    except KeyboardInterrupt:
        print("Monitoring stopped by user.")